[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/08_tag_2_3_skip_connections_functional_api.ipynb)

# Tag 2/3 - 08 Skip Connections mit Functional API

Skip Connections sollen den Gradientenfluss verbessern. Damit das Beispiel stabil bleibt, wird der Residual-Zweig leicht skaliert. Ohne diese Skalierung können sehr tiefe Residual-Blöcke am Anfang zu große Aktivierungen erzeugen.

Das Ziel dieser Demo: Das Modell mit Skip Connection soll **tatsächlich lernen** und im Verlauf mindestens so stabil sein wie das tiefe Modell ohne Skip.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


In [ ]:
X, y = make_moons(n_samples=1400, noise=0.30, random_state=RANDOM_STATE)
X = StandardScaler().fit_transform(X).astype("float32")
y = y.astype("float32")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=14, edgecolor="black", alpha=0.72)
plt.title("Schwieriger Two-Moons-Datensatz")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()


In [ ]:
def draw_skip_diagram():
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.axis("off")
    xs = [0.04, 0.20, 0.36, 0.52, 0.68, 0.84]
    labels = ["Input", "Stem", "Block 1", "Block 2", "Block 3", "output"]
    for x, label in zip(xs, labels):
        ax.add_patch(plt.Rectangle((x, 0.42), 0.11, 0.22, fill=False, linewidth=2))
        ax.text(x + 0.055, 0.53, label, ha="center", va="center", fontsize=9)
    for x1, x2 in zip(xs[:-1], xs[1:]):
        ax.annotate("", xy=(x2, 0.53), xytext=(x1 + 0.11, 0.53), arrowprops=dict(arrowstyle="->", linewidth=1.8))
    for x in [0.36, 0.52, 0.68]:
        ax.annotate(
            "Skip",
            xy=(x + 0.10, 0.64),
            xytext=(x - 0.03, 0.86),
            arrowprops=dict(arrowstyle="->", linewidth=1.8, connectionstyle="arc3,rad=-0.35"),
            ha="center",
            fontsize=8,
        )
    ax.set_title("Skip Connections: Block-Input wird später wieder addiert")
    plt.show()


draw_skip_diagram()


## Zwei tiefe Modelle

Beide Modelle verwenden denselben Optimizer. Der Unterschied ist der `layers.Add`-Block im Skip-Modell.

Der Residual-Zweig wird mit `0.1` skaliert. Das ist eine einfache Stabilisierung: Der Block startet näher an einer Identitätsabbildung und kann dann lernen, welche Korrektur zusätzlich sinnvoll ist.


In [ ]:
def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_plain_deep_model(depth=16, units=32):
    inputs = keras.Input(shape=(2,), name="features")
    x = layers.Dense(units, activation="relu", kernel_initializer="he_normal", name="stem")(inputs)
    for i in range(depth):
        # Normaler tiefer Pfad: Jede Schicht hängt komplett von der vorherigen ab.
        x = layers.Dense(units, activation="relu", kernel_initializer="he_normal", name=f"plain_dense_{i+1}")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)
    return compile_model(keras.Model(inputs, outputs, name="plain_deep_without_skip"))


def residual_block(x, units, block_id, residual_scale=0.1):
    skip = x
    residual = layers.Dense(units, activation="relu", kernel_initializer="he_normal", name=f"res_{block_id}_dense_1")(x)
    residual = layers.Dense(units, activation=None, kernel_initializer="he_normal", name=f"res_{block_id}_dense_2")(residual)
    # Parameterblock im Fokus: Residual-Zweig wird stabilisiert und dann addiert.
    residual = layers.Lambda(lambda tensor: residual_scale * tensor, name=f"res_{block_id}_scale")(residual)
    x = layers.Add(name=f"res_{block_id}_add")([skip, residual])
    x = layers.Activation("relu", name=f"res_{block_id}_relu")(x)
    return x


def build_skip_model(blocks=8, units=32):
    inputs = keras.Input(shape=(2,), name="features")
    x = layers.Dense(units, activation="relu", kernel_initializer="he_normal", name="stem")(inputs)
    for block_id in range(1, blocks + 1):
        x = residual_block(x, units=units, block_id=block_id)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)
    return compile_model(keras.Model(inputs, outputs, name="deep_with_skip_connections"))


tf.keras.utils.set_random_seed(RANDOM_STATE)
plain_model = build_plain_deep_model()
tf.keras.utils.set_random_seed(RANDOM_STATE)
skip_model = build_skip_model()

plain_model.summary()
skip_model.summary()


In [ ]:
histories = {}
for name, model in [("ohne_skip_tief", plain_model), ("mit_skip_tief", skip_model)]:
    history = model.fit(
        X_train,
        y_train,
        validation_split=0.25,
        epochs=80,
        batch_size=64,
        verbose=0,
    )
    train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    histories[name] = history
    print(
        f"{name:>14}: Train Loss={train_loss:.3f}, Train Accuracy={train_accuracy:.1%}, "
        f"Test Loss={test_loss:.3f}, Test Accuracy={test_accuracy:.1%}"
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, history in histories.items():
    axes[0].plot(history.history["loss"], linestyle="--", alpha=0.6, label=f"{name} train")
    axes[0].plot(history.history["val_loss"], linewidth=2, label=f"{name} val")
    axes[1].plot(history.history["accuracy"], linestyle="--", alpha=0.6, label=f"{name} train")
    axes[1].plot(history.history["val_accuracy"], linewidth=2, label=f"{name} val")
axes[0].set_title("Train- und Validation Loss")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Fehler / Loss")
axes[0].legend(fontsize=8)
axes[1].set_title("Train- und Validation Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
def plot_boundary(model, title, ax):
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, 170),
        np.linspace(X[:, 1].min() - 0.4, X[:, 1].max() + 0.4, 170),
    )
    grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
    proba = model.predict(grid, verbose=0).reshape(xx.shape)
    ax.contourf(xx, yy, proba, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.35)
    ax.contour(xx, yy, proba, levels=[0.5], colors="black")
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=12, edgecolor="black")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_boundary(plain_model, "Tief ohne Skip", axes[0])
plot_boundary(skip_model, "Tief mit Skip", axes[1])
plt.tight_layout()
plt.show()
